# D3 EC differentiation: gene-level guide permutation analysis

This notebook performs the guide-level statistical analysis described in the manuscript.

For every targeting guide, the input CSV must already contain the D3 endothelial-cell percentage calculated after excluding cells carrying both that targeting guide and any NT/SAFE control guide. The targeting-guide and NTC populations are therefore mutually exclusive.

For each gene:

1. The gene-level D3 EC percentage is the **unweighted arithmetic mean** across six targeting guides.
2. The reference is the unweighted arithmetic mean across **119 usable individual NTC guides**.
3. The differentiation effect is

\[
\log_2\left(
\frac{\text{mean D3 EC percentage across NTC guides}}
{\text{mean D3 EC percentage across targeting guides}}
\right).
\]

4. A two-sided guide-level permutation test compares the mean of the six targeting guides with the mean of the 119 NTC guides.
5. Labels are permuted 100,000 times while preserving group sizes of 6 and 119.
6. The empirical p-value is the proportion of permuted absolute mean differences at least as extreme as the observed absolute mean difference.
7. P-values are adjusted across all 298 genes using Benjamini–Hochberg.
8. Each guide is one unweighted replicate; guide cell counts are not used as statistical weights.

The notebook performs strict input validation and writes a guide-count QC table before testing.


## 1. Configuration

In [ ]:

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt


# ============================================================
# PATHS
# ============================================================

WORKDIR = Path(
    "/oak/stanford/groups/engreitz/Users/tri/CodeBase/"
    "programDE/notebooks/per_day_umaps"
)

INPUT_CSV = (
    WORKDIR
    / "d3_EC_percent_by_guide_with_individual_NT_excl_Hippo.csv"
)

OUTDIR = (
    WORKDIR
    / "d3_EC_guide_level_permutation_analysis"
)
OUTDIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = (
    OUTDIR
    / "d3_EC_gene_level_guide_permutation_results.csv"
)

TARGET_GUIDES_USED_PATH = (
    OUTDIR
    / "d3_EC_targeting_guides_used.csv"
)

NTC_GUIDES_USED_PATH = (
    OUTDIR
    / "d3_EC_119_NTC_guides_used.csv"
)

GUIDE_COUNT_QC_PATH = (
    OUTDIR
    / "d3_EC_guide_count_QC.csv"
)

MANIFEST_TEMPLATE_PATH = (
    OUTDIR
    / "six_guide_selection_manifest_template.csv"
)

PERMUTATION_INDICES_PATH = (
    OUTDIR
    / "permutation_target_indices_100000_seed_20260828.npz"
)

ANALYSIS_METADATA_PATH = (
    OUTDIR
    / "analysis_metadata.json"
)


# ============================================================
# ANALYSIS SETTINGS
# ============================================================

N_PERMUTATIONS = 100_000
RANDOM_SEED = 20260828
PERMUTATION_BATCH_SIZE = 10_000

EXPECTED_GENES = 298
EXPECTED_TARGET_GUIDES_PER_GENE = 6
EXPECTED_USABLE_NTC_GUIDES = 119

# Exact manuscript mode:
# every tested gene must have exactly six selected usable guides.
#
# The current library contains some target labels with guide counts other than
# six. If this validation fails, inspect GUIDE_COUNT_QC_PATH and supply a
# guide-selection manifest where applicable.
GUIDE_COUNT_MODE = "require_exactly_six"

# Optional CSV with columns:
#   guide_id
#   include_for_analysis
#
# Rows with include_for_analysis = TRUE are retained for targeting-guide tests.
# Leave as None to use every targeting-guide row in the input file.
GUIDE_SELECTION_MANIFEST_PATH = None

# This follows the requested definition literally:
# p = number of extreme permutations / 100,000.
#
# Set to True only for a sensitivity analysis using:
# (extreme + 1) / (N_PERMUTATIONS + 1).
USE_PLUS_ONE_EMPIRICAL_P = False

# The corrected upstream export should contain this value for targeting rows.
REQUIRED_TARGET_EXCLUSION_RULE = "exclude_NT_SAFE_cocarriers"

MAKE_QC_PLOTS = True


# ============================================================
# PLOT SETTINGS
# ============================================================

mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.transparent": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.family": "DejaVu Sans",
})

print("Input:", INPUT_CSV)
print("Output directory:", OUTDIR)


## 2. Statistical helper functions

In [ ]:

# ============================================================
# HELPERS
# ============================================================

def parse_bool_series(series):
    """
    Convert common TRUE/FALSE representations to bool.
    """
    if series.dtype == bool:
        return series

    true_values = {
        "true", "t", "1", "yes", "y", "include", "included"
    }
    false_values = {
        "false", "f", "0", "no", "n", "exclude", "excluded", ""
    }

    def convert(x):
        if pd.isna(x):
            return False

        value = str(x).strip().lower()

        if value in true_values:
            return True

        if value in false_values:
            return False

        raise ValueError(
            f"Could not interpret include_for_analysis value: {x!r}"
        )

    return series.map(convert).astype(bool)


def benjamini_hochberg(p_values):
    """
    Benjamini-Hochberg FDR adjustment.
    """
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(p.shape, np.nan, dtype=float)

    finite = np.isfinite(p)

    if finite.sum() == 0:
        return adjusted

    finite_p = p[finite]
    order = np.argsort(finite_p)
    ranked = finite_p[order]

    m = len(ranked)
    ranks = np.arange(1, m + 1, dtype=float)

    ranked_adjusted = ranked * m / ranks
    ranked_adjusted = np.minimum.accumulate(
        ranked_adjusted[::-1]
    )[::-1]
    ranked_adjusted = np.clip(
        ranked_adjusted,
        0,
        1,
    )

    unordered = np.empty_like(ranked_adjusted)
    unordered[order] = ranked_adjusted

    adjusted[finite] = unordered

    return adjusted


def fdr_to_stars(fdr):
    if not np.isfinite(fdr):
        return ""

    if fdr < 0.001:
        return "***"

    if fdr < 0.01:
        return "**"

    if fdr < 0.05:
        return "*"

    return ""


def generate_uniform_target_subsets(
    n_total,
    n_target,
    n_permutations,
    seed,
    batch_size=10_000,
):
    """
    Generate uniformly random target-index subsets without replacement.

    Each permutation assigns n_target of n_total guide values to the
    permuted target group. The remaining values form the permuted NTC group.

    Random priorities followed by argpartition produce an unbiased random
    subset of the requested size for each permutation.
    """
    if n_target <= 0 or n_target >= n_total:
        raise ValueError(
            f"Invalid group sizes: n_target={n_target}, n_total={n_total}"
        )

    rng = np.random.default_rng(seed)

    index_dtype = (
        np.uint8
        if n_total <= np.iinfo(np.uint8).max
        else np.uint16
    )

    result = np.empty(
        (n_permutations, n_target),
        dtype=index_dtype,
    )

    for start in range(0, n_permutations, batch_size):
        stop = min(
            start + batch_size,
            n_permutations,
        )
        batch_n = stop - start

        priorities = rng.random(
            (batch_n, n_total),
            dtype=np.float32,
        )

        chosen = np.argpartition(
            priorities,
            kth=n_target - 1,
            axis=1,
        )[:, :n_target]

        result[start:stop] = chosen.astype(
            index_dtype,
            copy=False,
        )

    return result


def permutation_test_one_gene(
    target_values,
    ntc_values,
    target_subset_indices,
    use_plus_one=False,
):
    """
    Two-sided guide-level permutation test.

    Observed statistic:
      mean(target guides) - mean(NTC guides)

    Permutation statistic:
      the same difference after permuting group labels while preserving
      target and NTC group sizes.
    """
    target_values = np.asarray(
        target_values,
        dtype=float,
    )
    ntc_values = np.asarray(
        ntc_values,
        dtype=float,
    )

    n_target = len(target_values)
    n_ntc = len(ntc_values)

    combined = np.concatenate([
        target_values,
        ntc_values,
    ])

    combined_sum = combined.sum()

    permuted_target_sums = combined[
        target_subset_indices
    ].sum(axis=1)

    permuted_target_means = (
        permuted_target_sums / n_target
    )

    permuted_ntc_means = (
        combined_sum - permuted_target_sums
    ) / n_ntc

    permuted_differences = (
        permuted_target_means
        - permuted_ntc_means
    )

    observed_target_mean = target_values.mean()
    observed_ntc_mean = ntc_values.mean()

    observed_difference = (
        observed_target_mean
        - observed_ntc_mean
    )

    # Numerical tolerance prevents a value equal within floating-point error
    # from being excluded.
    tolerance = 1e-12

    n_extreme = int(
        np.sum(
            np.abs(permuted_differences)
            >= abs(observed_difference) - tolerance
        )
    )

    n_perm = len(permuted_differences)

    if use_plus_one:
        p_value = (
            n_extreme + 1
        ) / (
            n_perm + 1
        )
    else:
        p_value = (
            n_extreme / n_perm
        )

    return {
        "mean_target_pct": observed_target_mean,
        "mean_ntc_pct": observed_ntc_mean,
        "observed_difference_target_minus_ntc": observed_difference,
        "n_extreme_permutations": n_extreme,
        "empirical_p_value": p_value,
    }


def log2_ntc_over_target(mean_ntc, mean_target):
    """
    Requested differentiation effect:
      log2(mean NTC EC percentage / mean target EC percentage)

    No pseudocount is added.
    """
    if not np.isfinite(mean_ntc) or not np.isfinite(mean_target):
        return np.nan

    if mean_ntc <= 0 or mean_target <= 0:
        return np.nan

    return float(
        np.log2(
            mean_ntc / mean_target
        )
    )


def write_manifest_template(targeting_guides):
    """
    Write a guide-selection template.

    Targets already having exactly six usable guides are pre-marked TRUE.
    Targets with other counts are left blank for manual review.
    """
    counts = targeting_guides.groupby(
        "target"
    )["guide_id"].transform("size")

    template = targeting_guides[
        [
            "target",
            "guide_id",
            "guide_name",
            "guide_index",
            "n_D3_cells",
            "pct_EC_D3",
        ]
    ].copy()

    template["n_usable_guides_for_target"] = counts

    template["include_for_analysis"] = np.where(
        counts == EXPECTED_TARGET_GUIDES_PER_GENE,
        "TRUE",
        "",
    )

    template = template.sort_values(
        ["target", "guide_index"],
        kind="stable",
    )

    template.to_csv(
        MANIFEST_TEMPLATE_PATH,
        index=False,
    )

    return template


## 3. Load the guide-level CSV and validate the six-guide/119-NTC design

In [ ]:

# ============================================================
# LOAD INPUT AND VALIDATE GUIDE-LEVEL VALUES
# ============================================================

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Input CSV was not found:\n{INPUT_CSV}\n\n"
        "Run the mutually exclusive query-guide/NT-SAFE export notebook first."
    )

df = pd.read_csv(INPUT_CSV)

required_columns = [
    "row_type",
    "guide_id",
    "guide_name",
    "guide_index",
    "target",
    "exclusion_rule",
    "n_D3_cells",
    "pct_EC_D3",
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Input CSV is missing required columns: {missing_columns}"
    )

df["n_D3_cells"] = pd.to_numeric(
    df["n_D3_cells"],
    errors="coerce",
)

df["pct_EC_D3"] = pd.to_numeric(
    df["pct_EC_D3"],
    errors="coerce",
)


# ------------------------------------------------------------------
# Targeting guides
# ------------------------------------------------------------------

targeting_all = df.loc[
    df["row_type"].eq("targeting_guide")
].copy()

# Require the corrected mutual-exclusion upstream output.
observed_rules = set(
    targeting_all["exclusion_rule"]
    .dropna()
    .astype(str)
    .unique()
)

if observed_rules != {REQUIRED_TARGET_EXCLUSION_RULE}:
    raise ValueError(
        "The targeting-guide rows do not have the expected mutual-exclusion "
        "rule.\n"
        f"Expected: {REQUIRED_TARGET_EXCLUSION_RULE!r}\n"
        f"Observed: {sorted(observed_rules)}\n\n"
        "Rerun the corrected upstream notebook that excludes NT/SAFE "
        "co-carriers from each query-guide group."
    )

targeting_usable = targeting_all.loc[
    targeting_all["pct_EC_D3"].notna()
    & targeting_all["n_D3_cells"].gt(0)
].copy()


# ------------------------------------------------------------------
# Optional six-guide manifest
# ------------------------------------------------------------------

if GUIDE_SELECTION_MANIFEST_PATH is not None:
    manifest_path = Path(
        GUIDE_SELECTION_MANIFEST_PATH
    )

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Guide-selection manifest not found:\n{manifest_path}"
        )

    manifest = pd.read_csv(manifest_path)

    required_manifest_columns = [
        "guide_id",
        "include_for_analysis",
    ]

    missing_manifest = [
        c for c in required_manifest_columns
        if c not in manifest.columns
    ]

    if missing_manifest:
        raise ValueError(
            "Guide-selection manifest is missing columns: "
            f"{missing_manifest}"
        )

    manifest["include_for_analysis"] = parse_bool_series(
        manifest["include_for_analysis"]
    )

    included_guide_ids = set(
        manifest.loc[
            manifest["include_for_analysis"],
            "guide_id",
        ].astype(str)
    )

    targeting_usable = targeting_usable.loc[
        targeting_usable["guide_id"]
        .astype(str)
        .isin(included_guide_ids)
    ].copy()

    print(
        "Applied guide-selection manifest:",
        manifest_path,
    )


# ------------------------------------------------------------------
# NTC guides
# ------------------------------------------------------------------

ntc_all = df.loc[
    df["row_type"].eq(
        "individual_NT_guide_excl_Hippo"
    )
].copy()

ntc_usable = ntc_all.loc[
    ntc_all["pct_EC_D3"].notna()
    & ntc_all["n_D3_cells"].gt(0)
].copy()

ntc_usable = ntc_usable.sort_values(
    "guide_index",
    kind="stable",
).reset_index(drop=True)


# ------------------------------------------------------------------
# Guide-count QC
# ------------------------------------------------------------------

guide_count_qc = (
    targeting_usable
    .groupby("target", as_index=False)
    .agg(
        n_usable_target_guides=(
            "guide_id",
            "size",
        ),
        min_D3_cells_per_guide=(
            "n_D3_cells",
            "min",
        ),
        max_D3_cells_per_guide=(
            "n_D3_cells",
            "max",
        ),
        mean_D3_EC_pct_unweighted=(
            "pct_EC_D3",
            "mean",
        ),
    )
    .sort_values("target")
    .reset_index(drop=True)
)

guide_count_qc[
    "has_exactly_six_guides"
] = (
    guide_count_qc[
        "n_usable_target_guides"
    ]
    == EXPECTED_TARGET_GUIDES_PER_GENE
)

guide_count_qc.to_csv(
    GUIDE_COUNT_QC_PATH,
    index=False,
)

targeting_usable.sort_values(
    ["target", "guide_index"],
    kind="stable",
).to_csv(
    TARGET_GUIDES_USED_PATH,
    index=False,
)

ntc_usable.to_csv(
    NTC_GUIDES_USED_PATH,
    index=False,
)

print("Usable individual NTC guides:", len(ntc_usable))
print("Tested target labels:", targeting_usable["target"].nunique())
print("\nTarget-guide count distribution:")
print(
    guide_count_qc[
        "n_usable_target_guides"
    ]
    .value_counts()
    .sort_index()
)

if len(ntc_usable) != EXPECTED_USABLE_NTC_GUIDES:
    raise ValueError(
        f"Expected {EXPECTED_USABLE_NTC_GUIDES} usable individual NTC guides, "
        f"but found {len(ntc_usable)}.\n"
        f"See: {NTC_GUIDES_USED_PATH}"
    )

if targeting_usable["target"].nunique() != EXPECTED_GENES:
    raise ValueError(
        f"Expected {EXPECTED_GENES} tested genes, but found "
        f"{targeting_usable['target'].nunique()}.\n"
        f"See: {GUIDE_COUNT_QC_PATH}"
    )

non_six = guide_count_qc.loc[
    ~guide_count_qc["has_exactly_six_guides"]
].copy()

if GUIDE_COUNT_MODE == "require_exactly_six" and len(non_six) > 0:
    template = write_manifest_template(
        targeting_usable
    )

    display(non_six)

    raise ValueError(
        "The exact manuscript analysis requires six usable targeting guides "
        "for every gene, but some target labels do not currently have six.\n\n"
        f"Guide-count QC: {GUIDE_COUNT_QC_PATH}\n"
        f"Manifest template: {MANIFEST_TEMPLATE_PATH}\n\n"
        "For targets with >6 guides, mark exactly six guide IDs TRUE in the "
        "manifest. Targets with <6 guides cannot satisfy the stated six-guide "
        "method from this CSV and require resolution before the exact "
        "298-gene analysis can be run."
    )

print("\nInput validation passed.")


## 4. Generate the 100,000 guide-label permutations

In [ ]:

# ============================================================
# GENERATE OR LOAD THE 100,000 LABEL PERMUTATIONS
# ============================================================

if GUIDE_COUNT_MODE != "require_exactly_six":
    raise ValueError(
        "This notebook is configured for the exact six-guide manuscript test."
    )

n_target_guides = EXPECTED_TARGET_GUIDES_PER_GENE
n_ntc_guides = len(ntc_usable)
n_total_guides = (
    n_target_guides
    + n_ntc_guides
)

if PERMUTATION_INDICES_PATH.exists():
    saved = np.load(
        PERMUTATION_INDICES_PATH
    )

    target_subset_indices = saved[
        "target_subset_indices"
    ]

    saved_seed = int(saved["seed"])
    saved_n_total = int(saved["n_total"])
    saved_n_target = int(saved["n_target"])
    saved_n_permutations = int(saved["n_permutations"])

    expected_shape = (
        N_PERMUTATIONS,
        EXPECTED_TARGET_GUIDES_PER_GENE,
    )

    if (
        target_subset_indices.shape != expected_shape
        or saved_seed != RANDOM_SEED
        or saved_n_total != n_total_guides
        or saved_n_target != n_target_guides
        or saved_n_permutations != N_PERMUTATIONS
    ):
        print(
            "[warning] Existing permutation-index file does not match the "
            "current settings. Regenerating."
        )

        target_subset_indices = generate_uniform_target_subsets(
            n_total=n_total_guides,
            n_target=n_target_guides,
            n_permutations=N_PERMUTATIONS,
            seed=RANDOM_SEED,
            batch_size=PERMUTATION_BATCH_SIZE,
        )

    else:
        print(
            "Loaded existing permutation indices:",
            PERMUTATION_INDICES_PATH,
        )

else:
    target_subset_indices = generate_uniform_target_subsets(
        n_total=n_total_guides,
        n_target=n_target_guides,
        n_permutations=N_PERMUTATIONS,
        seed=RANDOM_SEED,
        batch_size=PERMUTATION_BATCH_SIZE,
    )

np.savez_compressed(
    PERMUTATION_INDICES_PATH,
    target_subset_indices=target_subset_indices,
    seed=np.array(RANDOM_SEED),
    n_total=np.array(n_total_guides),
    n_target=np.array(n_target_guides),
    n_permutations=np.array(N_PERMUTATIONS),
)

print("Permutation index shape:", target_subset_indices.shape)
print("Saved permutation indices:", PERMUTATION_INDICES_PATH)


## 5. Run the gene-level tests and BH correction

In [ ]:

# ============================================================
# RUN THE 298 GENE-LEVEL PERMUTATION TESTS
# ============================================================

ntc_values = ntc_usable[
    "pct_EC_D3"
].to_numpy(dtype=float)

mean_ntc_pct = float(
    ntc_values.mean()
)

results = []

grouped = targeting_usable.groupby(
    "target",
    sort=True,
)

for target, group in grouped:
    group = group.sort_values(
        "guide_index",
        kind="stable",
    )

    target_values = group[
        "pct_EC_D3"
    ].to_numpy(dtype=float)

    if len(target_values) != EXPECTED_TARGET_GUIDES_PER_GENE:
        raise ValueError(
            f"{target}: expected six guides but found {len(target_values)}"
        )

    test = permutation_test_one_gene(
        target_values=target_values,
        ntc_values=ntc_values,
        target_subset_indices=target_subset_indices,
        use_plus_one=USE_PLUS_ONE_EMPIRICAL_P,
    )

    effect = log2_ntc_over_target(
        mean_ntc=test["mean_ntc_pct"],
        mean_target=test["mean_target_pct"],
    )

    results.append({
        "target": target,
        "n_target_guides": len(target_values),
        "n_NTC_guides": len(ntc_values),
        "mean_D3_EC_pct_target_guides": test["mean_target_pct"],
        "sd_D3_EC_pct_target_guides": float(
            np.std(
                target_values,
                ddof=1,
            )
        ),
        "mean_D3_EC_pct_NTC_guides": test["mean_ntc_pct"],
        "sd_D3_EC_pct_NTC_guides": float(
            np.std(
                ntc_values,
                ddof=1,
            )
        ),
        "log2_effect_NTC_over_target": effect,
        "observed_mean_difference_target_minus_NTC_pct_points": (
            test[
                "observed_difference_target_minus_ntc"
            ]
        ),
        "abs_observed_mean_difference_pct_points": abs(
            test[
                "observed_difference_target_minus_ntc"
            ]
        ),
        "n_permutations": N_PERMUTATIONS,
        "n_extreme_permutations": test[
            "n_extreme_permutations"
        ],
        "empirical_p_value": test[
            "empirical_p_value"
        ],
        "target_guide_ids": ";".join(
            group["guide_id"].astype(str)
        ),
        "target_guide_EC_percentages": ";".join(
            f"{x:.10g}"
            for x in target_values
        ),
    })

results_df = pd.DataFrame(results)

if len(results_df) != EXPECTED_GENES:
    raise ValueError(
        f"Expected {EXPECTED_GENES} gene results, found {len(results_df)}."
    )

results_df["FDR_BH"] = benjamini_hochberg(
    results_df["empirical_p_value"]
)

results_df["significance"] = results_df[
    "FDR_BH"
].map(fdr_to_stars)

results_df["effect_direction"] = np.select(
    [
        results_df[
            "log2_effect_NTC_over_target"
        ].gt(0),
        results_df[
            "log2_effect_NTC_over_target"
        ].lt(0),
    ],
    [
        "targeting guides reduce EC percentage",
        "targeting guides increase EC percentage",
    ],
    default="no change",
)

results_df = results_df.sort_values(
    [
        "FDR_BH",
        "empirical_p_value",
        "abs_observed_mean_difference_pct_points",
    ],
    ascending=[
        True,
        True,
        False,
    ],
    kind="stable",
).reset_index(drop=True)

results_df["rank_by_FDR"] = np.arange(
    1,
    len(results_df) + 1,
)

# Move rank close to the front.
front_columns = [
    "rank_by_FDR",
    "target",
    "n_target_guides",
    "n_NTC_guides",
    "mean_D3_EC_pct_target_guides",
    "mean_D3_EC_pct_NTC_guides",
    "log2_effect_NTC_over_target",
    "observed_mean_difference_target_minus_NTC_pct_points",
    "empirical_p_value",
    "FDR_BH",
    "significance",
    "effect_direction",
]

other_columns = [
    c for c in results_df.columns
    if c not in front_columns
]

results_df = results_df[
    front_columns + other_columns
]

results_df.to_csv(
    RESULTS_PATH,
    index=False,
)

print("Saved results:", RESULTS_PATH)
print("Results:", len(results_df))
print("FDR < 0.05:", int((results_df["FDR_BH"] < 0.05).sum()))
print("FDR < 0.01:", int((results_df["FDR_BH"] < 0.01).sum()))
print("FDR < 0.001:", int((results_df["FDR_BH"] < 0.001).sum()))

display(
    results_df.head(20)
)


## 6. Save analysis metadata and QC plots

In [ ]:

# ============================================================
# METADATA AND OPTIONAL QC PLOTS
# ============================================================

metadata = {
    "input_csv": str(INPUT_CSV),
    "results_path": str(RESULTS_PATH),
    "analysis_unit": "guide",
    "guide_weighting": "unweighted",
    "target_guides_per_gene": EXPECTED_TARGET_GUIDES_PER_GENE,
    "usable_individual_NTC_guides": len(ntc_usable),
    "tested_genes": len(results_df),
    "effect_definition": (
        "log2(mean D3 EC percentage across NTC guides / "
        "mean D3 EC percentage across targeting guides)"
    ),
    "observed_test_statistic": (
        "mean target-guide EC percentage minus mean NTC-guide EC percentage"
    ),
    "test": "two-sided guide-label permutation test",
    "n_permutations": N_PERMUTATIONS,
    "random_seed": RANDOM_SEED,
    "empirical_p_definition": (
        "(number of permutations with absolute mean difference >= observed "
        "absolute mean difference) / N_PERMUTATIONS"
        if not USE_PLUS_ONE_EMPIRICAL_P
        else "(extreme + 1) / (N_PERMUTATIONS + 1)"
    ),
    "multiple_testing": (
        "Benjamini-Hochberg across all tested genes"
    ),
    "targeting_guide_exclusion_rule_required": (
        REQUIRED_TARGET_EXCLUSION_RULE
    ),
    "significance_thresholds": {
        "*": "FDR < 0.05",
        "**": "FDR < 0.01",
        "***": "FDR < 0.001",
    },
}

with open(
    ANALYSIS_METADATA_PATH,
    "w",
) as handle:
    json.dump(
        metadata,
        handle,
        indent=2,
    )

print("Saved metadata:", ANALYSIS_METADATA_PATH)


if MAKE_QC_PLOTS:
    # P-value histogram
    fig, ax = plt.subplots(
        figsize=(5.0, 3.8)
    )

    ax.hist(
        results_df["empirical_p_value"],
        bins=np.linspace(0, 1, 21),
        edgecolor="black",
        linewidth=0.7,
    )

    ax.set_xlabel(
        "Empirical permutation p-value"
    )
    ax.set_ylabel(
        "Number of genes"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()

    p_hist_pdf = (
        OUTDIR
        / "QC_empirical_p_value_histogram.pdf"
    )
    p_hist_png = (
        OUTDIR
        / "QC_empirical_p_value_histogram.png"
    )

    fig.savefig(
        p_hist_pdf,
        bbox_inches="tight",
    )
    fig.savefig(
        p_hist_png,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    # Effect versus FDR
    fig, ax = plt.subplots(
        figsize=(5.0, 4.2)
    )

    plot_fdr = results_df[
        "FDR_BH"
    ].clip(lower=1 / N_PERMUTATIONS)

    ax.scatter(
        results_df[
            "log2_effect_NTC_over_target"
        ],
        -np.log10(plot_fdr),
        s=18,
        alpha=0.7,
        edgecolors="none",
    )

    ax.axhline(
        -np.log10(0.05),
        linestyle="--",
        linewidth=1,
        color="black",
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1,
        color="gray",
    )

    ax.set_xlabel(
        "log2(mean NTC EC% / mean target EC%)"
    )
    ax.set_ylabel(
        "-log10(BH FDR)"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()

    volcano_pdf = (
        OUTDIR
        / "QC_effect_vs_FDR.pdf"
    )
    volcano_png = (
        OUTDIR
        / "QC_effect_vs_FDR.png"
    )

    fig.savefig(
        volcano_pdf,
        bbox_inches="tight",
    )
    fig.savefig(
        volcano_png,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    print("Saved QC plots to:", OUTDIR)
